In [18]:
# rag pipeline data ingestion to vector pipeline

In [1]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


/var/folders/cf/8zyqpdhs1l900bf7ll9jp8180000gp/T/ipykernel_45827/2452563033.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
## Read all pdfs in dirs
def process_all_pdfs(pdf_directory):
    
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nprocessing: {pdf_file.name}")
        
        try:
            
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
                
        except Exception as e:
            print(f"X Error: {e}")
    
    return all_documents
        

all_pdf_documents = process_all_pdfs("../data") 

Found 2 PDF files to process

processing: 2312.10997v5.pdf
Loaded 21 pages

processing: 2506.18027v3.pdf
Loaded 14 pages


In [21]:
# all_pdf_documents

In [3]:
def document_chunking(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunk for better rag performances"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ","",","]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [4]:
chunks=document_chunking(all_pdf_documents)

Split 35 documents into 201 chunks

Example chunk:
Content: 1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng
Wangc, and Haofen Wan
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-03-28T00:54:45+00:00', 'author': '', 'keywords': '', 'moddate': '2024-03-28T00:54:45+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/2312.10997v5.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1', 'source_file': '2312.10997v5.pdf', 'file_type': 'pdf'}


In [5]:
# embedding and vectorDB

import uuid
from typing import Any, Dict, List, Tuple

import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


/Users/primesysindia/priyanshuprimesys/PersonalFolder/aicourses/rag_project/trag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1025 15:47:05.224000 45827 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [6]:
class EmbeddingManager:
    """ Handle document embedding generation using Sentence Transformers """
    
    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
            Initialize the embedding manager
        Args:
            model_name: Hugging face model for sentence embeddings
        """
        
        self.model_name= model_name
        self.model= None
        self._load_model()
        
    def _load_model(self):
        """Load the sentence transformer model"""
        
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self,texts: list[str]) -> np.ndarray:
        """ 
        Generate embedding for a list of texts
        
        Args:
            texts: List of texts string to embed
        Return:
            numpy array of embeddings with shape (len(texts),embedding_dim)
        """
        
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generate Embeddings for {len(texts)} texts....")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated Embeddings with shape: {embeddings.shape}")
        return embeddings
    
    
    
    
    
# intialize embedding manager
embedding_manager = EmbeddingManager()
embedding_manager
        

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11829.82it/s]


Model Loaded successfully. Embedding dimension: 384


In [26]:
## Vector Store

In [7]:
class VectorStore:
    """ Manage document embeddings in ChromaDB vector store"""
    
    def __init__(self,collection_name: str="pdf_document",persist_directory:str="../data/vector_store"):
        """ 
        Initialize the vector store
        
        Args:
            collection_name: Name of chromaDB collection
            persist_directory: Directory to persist the vector store
        """
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    
    def _initialize_store(self):
        """ Initialize chromadb and collection """
        
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.create_collection(
                name=self.collection_name,
                metadata={
                    "description":"PDF documents embeddings for rag"
                }
            )
            
            print(f"Vector store intialized. collection {self.collection_name}")
            print(f"Existing documents in collection {self.collection.count()}")
            
        except Exception as e:
            print(f"Error intitializing vector store: {e}")
            raise
        
    def add_documents(self,documents: list[Any], embeddings: np.ndarray):
        
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        
        print(f"Adding {len(documents)} to vector store")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare meta data
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
            
            
        # Add to collection
        
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            
            print(f"Successfully added {len(documents)} to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise 
        
        
        
        
        
        
vectorstore = VectorStore()


Vector store intialized. collection pdf_document
Existing documents in collection 0


In [8]:
## Convert the text to embeddings

texts=[doc.page_content for doc in chunks]

## Generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts)


## store inside vector store
vectorstore.add_documents(chunks,embeddings)

Generate Embeddings for 201 texts....


Batches: 100%|██████████| 7/7 [00:01<00:00,  6.25it/s]


Generated Embeddings with shape: (201, 384)
Adding 201 to vector store
Successfully added 201 to vector store
Total documents in collection: 201


In [9]:
## RAG Retriever

class RAGRetriever:
    """ Handles everything query-based retrieval from the vector store """
    
    def __init__(self, vectorstore: VectorStore, embeddingManager: EmbeddingManager):
        self.vectorstore = vectorstore
        self.embeddingManager = embeddingManager
        
    
    def retrieve(self, query:str,top_k:int=5,score_threshold: float = 0.0)-> List[dict[str,Any]]:
        """ Retrieve the relevant documents for a query 
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity threshold

        Returns:
            List of documents containing retrieved documents and metadata
        """
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score_threshold: {score_threshold}")
        
        
        # Generate query embeddings
        
        query_embedding = self.embeddingManager.generate_embeddings([query])[0]
        
        # Search in a vector store
        
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            
            # processed results
            
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    
                    similariy_score = 1 + distance
                     
                    if similariy_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similariy_score,
                            'distance':distance,
                            'rank':i+1
                        })
                        
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                
            else:
                print("No document found")
                
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval : {e}")
            return []
        
        
        
        
        
rag_retriver = RAGRetriever(vectorstore,embedding_manager)

                    

In [10]:

rag_retriver

In [11]:
rag_retriver.retrieve("What is machine learning")

Retrieving documents for query: 'What is machine learning'
Top K: 5, score_threshold: 0.0
Generate Embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.49it/s]

Generated Embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_15475df5_53',
  'content': 'purpose of fine-tuning is to align the retriever and generator,\nfor example, using the results of LLM as the supervision signal\nfor fine-tuning, known as LSR (LM-supervised Retriever).\nPROMPTAGATOR [21] utilizes the LLM as a few-shot query\ngenerator to create task-specific retrievers, addressing chal-\nlenges in supervised fine-tuning, particularly in data-scarce\ndomains. Another approach, LLM-Embedder [97], exploits\nLLMs to generate reward signals across multiple downstream\ntasks. The retriever is fine-tuned with two types of supervised\nsignals: hard labels for the dataset and soft rewards from\nthe LLMs. This dual-signal approach fosters a more effective\nfine-tuning process, tailoring the embedding model to diverse\ndownstream applications. REPLUG [72] utilizes a retriever\nand an LLM to calculate the probability distributions of the\nretrieved documents and then performs supervised training\nby computing the KL divergence. This strai

In [12]:
rag_retriver.retrieve("What is embedding")

Retrieving documents for query: 'What is embedding'
Top K: 5, score_threshold: 0.0
Generate Embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.70it/s]

Generated Embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_b2e7aa71_161',
  'content': 'include these references, ensuring that images are identified and described. During markdown\nconversion, tables are represented in markdown syntax initially which can be verbose and inefficient.\nTo enhance this, we compress markdown tables as dictionary format for efficient storage, reducing\nthe storage space leading to easier data access, understanding and manipulation.\n3.2.2 Embedding and storage\nThe first step involves breaking down the processed document ˆDimarkdown into smaller chunks\n{c1, c2, ...cm}. Data consisting of text, image captions and dictionary-formatted tables into separate\nsegments. Each chunk Cj is embedded into a high-dimensional vector space using an embedding\nmodelf embed:\nfembed(cj)→v j ∈R d (3)\nThese embeddings are then stored in a searchable database (Elasticsearch):\nElasticsearch(v 1,v 2, ...vm)(4)\nTo enhance the retrieval process, we employ Recursive Abstractive Processing for Tree-Organized',
  'metadata'

In [29]:
## Integration vectordb context pipeline with LLM Output


from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv()


llm = ChatOllama(
    model="qwen3.5:2b",
    base_url="http://localhost:11434",
    temperature=0,
    reasoning=False,
    num_predict=256
)

# Simple RAG function
def rag_simple(query,retriever,llm,top_k=3):
    
    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    
    
    # generate the answer using the ollama
    prompt=f"""Use the following context to answer the question concisely.
        Context: 
        {context}
        
        Question: {query}
        
        Answer:"""
        
    
        
    response = llm.invoke([prompt.format(context=context,query=query)])
    
    
    return response.content
    

In [30]:
# Answer testing
answer=rag_simple("What is machine learning?",rag_retriver,llm,top_k=3)
print(answer)

Retrieving documents for query: 'What is machine learning?'
Top K: 3, score_threshold: 0.0
Generate Embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.18it/s]

Generated Embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Based on the provided context, **machine learning** (specifically in this text) refers to a process where an algorithm or model learns from data and supervision signals—such as hard labels for datasets or soft rewards from LLMs—to align with specific goals. It is used to fine-tune retrievers and generators into task-specific models that can adapt to diverse downstream applications, handle structured data formats (via frameworks like SANTA), generate responses in particular styles, and optimize performance through methods like contrastive learning and reinforcement learning.
